# ATiG 2026: LD score regression
29 October 2026 · About 60 minutes · Work in pairs

We use LD score regression (LDSC) to estimate heritability and genetic correlations from
published GWAS results. Run the cells in order and discuss the questions.
[Open in Colab](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-10-29/LDSC/LDSC_practical.ipynb) and choose File → Save a copy in Drive.

## Setup
Run the three cells. The last one downloads the three GWAS from the
[GWAS Catalog](https://www.ebi.ac.uk/gwas/) (about 260 MB, a few minutes).

In [ ]:
%pip install -q "allelign>=0.2.0" "pldsc @ git+https://github.com/bvilhjal/PLDSC@v0.1.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import allelign
from pldsc import load_ld_reference, ldsc_h2_sumstats, ldsc_rg_sumstats

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
#@title Download the GWAS files from the GWAS Catalog (no editing needed)
STUDIES = {'BMI': 'GCST006900', 'TG': 'GCST002216', 'DBP': 'GCST006630'}

paths = {}
for trait, study in STUDIES.items():  # A few minutes the first time.
    paths[trait] = allelign.fetch_ref(study)
    print(trait, study, paths[trait].name)

## Q1. Look at the files

The GWAS are of BMI ([Yengo et al. 2018](https://pubmed.ncbi.nlm.nih.gov/30124842/)),
triglycerides, TG ([Willer et al. 2013](https://pubmed.ncbi.nlm.nih.gov/24097068/)), and diastolic
blood pressure, DBP ([Evangelou et al. 2018](https://pubmed.ncbi.nlm.nih.gov/30224653/)). The BMI and
DBP files are as the authors submitted them. The GWAS Catalog has harmonised the TG file,
adding columns that start with `hm_`.

Find the effect allele, the effect size and its standard error in each file. How do the
files differ?

In [ ]:
for trait, path in paths.items():
    print(f'The first rows of the {trait} file:')
    display(pd.read_csv(path, sep=r'\s+', nrows=3, dtype=str))

## Q2. Read the files

allelign reads all three formats (Table 1) and takes DBP's positions from `MarkerName`.
The harmonised TG file has positions on genome build GRCh38, but our LD reference uses
GRCh37, so we match TG by rsID instead.

Which columns were not found in each file?

In [ ]:
columns = {}
for trait, path in paths.items():
    columns[trait] = allelign.inspect_columns(path)['mapping']

print('Table 1. File columns used by allelign; blank if not found.')
display(pd.DataFrame(columns).fillna(''))

In [ ]:
def read_gwas(path):
    """Read a GWAS file. Harmonised GWAS Catalog files (*.h.tsv.gz) have GRCh38
    positions, and our LD reference is GRCh37, so for those we match by rsID only."""
    sumstats = allelign.read_sumstats(path)
    if path.name.endswith('.h.tsv.gz'):
        sumstats.chrom[:] = ''
        sumstats.pos[:] = 0
    return sumstats


gwas = {}
for trait, path in paths.items():  # About 30 seconds.
    gwas[trait] = read_gwas(path)
    print(f'{trait}: {len(gwas[trait]):,} variants')

## Q3. Match to the LD reference

PLDSC comes with LD scores for about 1.2 million HapMap3 SNPs, from Europeans in the 1000
Genomes Project. We keep only these SNPs and flip each GWAS to the same allele, `a1`
(Table 2). Table 3 shows the result for BMI.

Why must all traits use the same allele? Allele C of rs1421085 in *FTO* raises BMI: does
it raise TG and DBP? Which trait has the most associated regions in Figure 1?

In [ ]:
ref = load_ld_reference('hm3-eur')

aligned = {}
counts = {}
for trait, sumstats in gwas.items():
    keep, qc = allelign.qc_sumstats(sumstats)
    aligned[trait] = allelign.harmonize(sumstats.subset(keep), ref.variants)
    counts[trait] = {'in file': len(sumstats),
                     'matched': aligned[trait].log['n_matched'],
                     'flipped': aligned[trait].log['n_flipped']}

print('Table 2. Variants in each file, matched to the reference, and flipped to a1.')
display(pd.DataFrame(counts))

In [ ]:
snps = {}
for trait, h in aligned.items():
    rows = h.var_index  # Rows of the LD reference.
    z = h.beta / h.se
    snps[trait] = pd.DataFrame({'snp': ref.variants.id[rows],
                                'chrom': ref.variants.chrom[rows].astype(int),
                                'pos': ref.variants.pos[rows],
                                'a1': ref.variants.a1[rows],
                                'z': z,
                                'chi2': z**2,
                                'p': 2 * stats.norm.sf(np.abs(z)),
                                'n': h.n_eff.astype(int),
                                'ld_score': ref.scores[rows]})

print('Table 3. The first aligned SNPs for BMI (z is for allele a1).')
display(snps['BMI'].head())

In [ ]:
snp = 'rs1421085'  # TRY: rs1260326 (GCKR) or rs13107325 (SLC39A8)
for trait, table in snps.items():
    row = table[table['snp'] == snp]
    print(f"{trait}: z = {row['z'].item():.2f} for allele {row['a1'].item()}")

In [ ]:
def manhattan(ax, table):
    """Plot the 1% of SNPs with the smallest P values along the genome."""
    top = table.nsmallest(len(table) // 100, 'p')
    x = top['chrom'] + top['pos'] / 250e6        # Chromosomes side by side.
    y = -np.log10(top['p'].clip(lower=1e-50))     # P below 1e-50 is drawn at 50.
    colour = np.where(top['chrom'] % 2 == 0, 'grey', 'steelblue')
    ax.scatter(x, y, c=colour, s=4)
    ax.axhline(-np.log10(5e-8), color='red')      # Genome-wide significance.
    ax.set_xticks(range(1, 23))
    ax.set_ylabel('-log10 P')


fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, trait in zip(axes, snps):
    manhattan(ax, snps[trait])
    ax.set_title(trait)
axes[-1].set_xlabel('Chromosome')
fig.suptitle('Figure 1. The strongest 1% of SNPs; red line: P = 5e-8')
plt.tight_layout()
plt.show()

## Q4. Heritability

LDSC regresses each SNP's χ² on its LD score $\ell_j$:

$$E[\chi^2_j] = 1 + Na + \frac{N h^2}{M}\,\ell_j,$$

where $N$ is the sample size and $M$ the number of SNPs. The slope gives the heritability
$h^2$, and the intercept $1 + Na$ measures confounding (Table 4, Figure 2).

Which trait is most heritable? Is there much confounding? Set `M = ref.m_snps` and rerun:
what changes?

In [ ]:
M = ref.m_snps_5_50  # SNPs with allele frequency 5-50%. TRY: ref.m_snps

fits = {}
h2_table = {}
for trait, sumstats in gwas.items():  # About 30 seconds.
    fit = ldsc_h2_sumstats(sumstats, ref.variants, ref.scores, m_snps=M).estimate
    fits[trait] = fit
    h2_table[trait] = {'h2': fit.h2, 'SE': fit.h2_se, 'intercept': fit.intercept,
                       'lambda GC': allelign.genomic_inflation(z=snps[trait]['z'].to_numpy())}

print(f'Table 4. SNP heritability with M = {M:,}.')
display(pd.DataFrame(h2_table).T)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, trait in zip(axes, snps):
    table = snps[trait]
    table = table[table['chi2'] <= np.maximum(80, table['n'] / 1000)]  # The SNPs LDSC uses.
    bins = pd.qcut(table['ld_score'], 50)                              # 50 bins of equal size.
    means = table.groupby(bins, observed=True)[['ld_score', 'chi2']].mean()
    ax.scatter(means['ld_score'], means['chi2'], s=10)

    fit = fits[trait]
    slope = table['n'].median() * fit.h2 / M
    ax.axline((0, fit.intercept), slope=slope, color='darkorange')   # The LDSC fit.
    ax.set_title(trait)
    ax.set_xlabel('LD score')
axes[0].set_ylabel('Mean χ²')
fig.suptitle('Figure 2. Mean χ² in LD-score bins, with the LDSC fit (orange)')
plt.tight_layout()
plt.show()

## Q5. Genetic correlations

Bivariate LDSC regresses the product of two traits' z scores on LD score. The slope, scaled
by the two heritabilities, gives the genetic correlation $r_g$, and the intercept reflects
people who are in both GWAS (Table 5, Figure 3).

Which traits are genetically correlated? UK Biobank is in both the BMI and the DBP GWAS, and
DBP was adjusted for BMI. Can you explain the BMI–DBP result?

In [ ]:
rg_table = {}
for trait1, trait2 in [('BMI', 'TG'), ('BMI', 'DBP'), ('TG', 'DBP')]:  # About a minute.
    fit = ldsc_rg_sumstats(gwas[trait1], gwas[trait2], ref.variants, ref.scores,
                           m_snps=M, ncores=2).estimate
    rg_table[f'{trait1}–{trait2}'] = {'rg': fit.rg, 'SE': fit.rg_se, 'intercept': fit.gcov_intercept}

results = pd.DataFrame(rg_table).T
print('Table 5. Genetic correlations.')
display(results)

plt.figure(figsize=(5, 2.5))
plt.errorbar(results['rg'], results.index, xerr=1.96 * results['SE'], fmt='o')
plt.gca().invert_yaxis()  # Same order as Table 5.
plt.axvline(0, color='grey', linestyle='--')
plt.xlabel('Genetic correlation, with 95% interval')
plt.title('Figure 3. Genetic correlations')
plt.tight_layout()
plt.show()

## Q6. Conclude

Write three sentences: the BMI–TG $r_g$ with its standard error, why it does not show that
BMI raises TG, and what the BMI–DBP result teaches you about published GWAS.

## Going further (optional)

- Try other traits. Find a study with full summary statistics in the
  [GWAS Catalog](https://www.ebi.ac.uk/gwas/downloads/summary-statistics) and put its accession
  in the cell below. European-ancestry GWAS fit our LD reference best.
- Ask an AI model to write LDSC from scratch in NumPy, using the columns `chi2`, `n` and
  `ld_score` of `snps['BMI']`, and compare with Table 4.

In [ ]:
study = 'GCST002223'  # HDL. TRY: GCST002222 (LDL), GCST006901 (height) or another study

new = read_gwas(allelign.fetch_ref(study))
print(study, ldsc_h2_sumstats(new, ref.variants, ref.scores, m_snps=M).estimate)
for trait in ['BMI', 'TG', 'DBP']:
    fit = ldsc_rg_sumstats(gwas[trait], new, ref.variants, ref.scores,
                           m_snps=M, ncores=2).estimate
    print(f'{trait}–{study}: rg {fit.rg:.3f} (SE {fit.rg_se:.3f})')

## Data and methods

All three GWAS are European-ancestry meta-analyses from the GWAS Catalog
([GCST006900](https://www.ebi.ac.uk/gwas/studies/GCST006900),
[GCST002216](https://www.ebi.ac.uk/gwas/studies/GCST002216),
[GCST006630](https://www.ebi.ac.uk/gwas/studies/GCST006630); see
[`data/sources.json`](data/sources.json)), downloaded with allelign. The LD scores are the 1000 Genomes European scores
released with LDSC ([Zenodo 7768714](https://doi.org/10.5281/zenodo.7768714)); PLDSC leaves
out about 30,000 SNPs in long-range LD regions such as the MHC, and SNPs with χ² above
max(80, N/1000). Background: [Bulik-Sullivan et al. 2015a](https://pubmed.ncbi.nlm.nih.gov/25642630/)
and [2015b](https://pubmed.ncbi.nlm.nih.gov/26414676/).